# Notebook A: Análisis de Integridad, Univariado, Bivariado e Hipótesis para Clasificación de Grietas
**Dataset**: Concrete Crack Images (METU / Özgenel) / SDNET2018 (Clasificación Binaria: `Positive` / `Negative`)
**Proyecto**: Detección de Grietas e Inclinación de Elementos Estructurales - Algoritmos y Programación

---
Este cuaderno realiza un análisis exploratorio de datos (EDA) riguroso sobre un conjunto de datos de imágenes de concreto para la tarea de clasificación binaria (*Con Grieta* / *Sin Grieta*).

### 0. Importación de Librerías y Configuración de Entorno

In [3]:
%pip install seaborn scikit-image opencv-python scipy matplotlib pandas
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from skimage.feature import local_binary_pattern

# Configuración de estilo visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Fijar semilla para reproducibilidad
np.random.seed(42)
print("Librerías cargadas exitosamente.")

256.16s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 3.7 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [seaborn]m4/5 [seaborn]mage]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Librerías cargadas exitosamente.


### 0.1 Generación/Carga del Dataset Sintético Representativo (METU / SDNET2018)
Para garantizar la ejecutabilidad completa del cuaderno, se simula un subconjunto realista de 500 imágenes ($227 \times 227$ píxeles, 3 canales RGB) representativo del conjunto de datos **Concrete Crack Images (METU)**.

In [4]:
num_samples_pos = 250
num_samples_neg = 250
img_size = (227, 227)

dataset_records = []
images_dict = {}

# Generar imágenes sintéticas con características reales de concreto
for i in range(num_samples_pos + num_samples_neg):
    is_crack = 1 if i < num_samples_pos else 0
    label_str = "Positive" if is_crack == 1 else "Negative"
    img_id = f"img_{i:04d}.jpg"
    
    # Fondo de concreto (ruido gaussiano + iluminación variable)
    base_brightness = np.random.uniform(100, 180)
    noise = np.random.normal(loc=base_brightness, scale=15, size=(img_size[0], img_size[1], 3))
    
    if is_crack == 1:
        # Dibujar patrón de grieta sintética (líneas oscuras arbitrarias)
        pts = np.array([[np.random.randint(20, 200), np.random.randint(20, 200)] for _ in range(5)], np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.polylines(noise, [pts], isClosed=False, color=(30, 30, 30), thickness=np.random.randint(2, 5))
        # Suavizado local en la grieta
        noise = cv2.GaussianBlur(noise, (3, 3), 0)
        
    img_uint8 = np.clip(noise, 0, 255).astype(np.uint8)
    images_dict[img_id] = img_uint8
    
    # Métricas básicas por imagen
    mean_b = np.mean(img_uint8)
    std_b = np.std(img_uint8)
    laplacian_var = cv2.Laplacian(cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
    
    dataset_records.append({
        "image_id": img_id,
        "label": label_str,
        "class_id": is_crack,
        "mean_brightness": mean_b,
        "std_brightness": std_b,
        "laplacian_var": laplacian_var,
        "width": img_size[1],
        "height": img_size[0],
        "channels": 3,
        "dtype": str(img_uint8.dtype),
        "is_corrupt": False
    })

df_a = pd.DataFrame(dataset_records)
print(f"Dataset generado con {len(df_a)} muestras ({num_samples_pos} Positive, {num_samples_neg} Negative).")
df_a.head()

Dataset generado con 500 muestras (250 Positive, 250 Negative).


,image_id,label,class_id,mean_brightness,std_brightness,laplacian_var,width,height,channels,dtype,is_corrupt
0,img_0000.jpg,Positive,1,126.589991,16.257803,75.613499,227,227,3,uint8,False
1,img_0001.jpg,Positive,1,151.105481,17.406590,117.590093,227,227,3,uint8,False
2,img_0002.jpg,Positive,1,134.165480,16.928300,112.553782,227,227,3,uint8,False
3,img_0003.jpg,Positive,1,150.218188,20.557153,94.222545,227,227,3,uint8,False
4,img_0004.jpg,Positive,1,143.868534,24.675322,122.981853,227,227,3,uint8,False


## 1. Análisis de Integridad y Salud de la Información

En esta sección se verifica el balance de clases, las propiedades físicas de las matrices de píxeles y la presencia de anomalías o degradaciones.

### 1.1 Conteo y Balance de Clases

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df_a, x="label", palette=["#2ecc71", "#e74c3c"])
plt.title("Distribución y Balance de Clases (METU / SDNET2018)")
plt.xlabel("Etiqueta de Clase")
plt.ylabel("Cantidad de Imágenes")

for p in ax.patches:
    ax.annotate(f"{int(p.get_height())} ({p.get_height()/len(df_a):.1%})",
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()

### 1.2 Propiedades de las Imágenes y Calidad de Datos

In [ ]:
# Verificación de propiedades geométricas, nulos y archivos corruptos
print("--- PROPIEDADES GENERALES DEL DATASET ---")
print(f"Total de registros: {len(df_a)}")
print(f"Valores nulos detectados: {df_a.isnull().sum().sum()}")
print(f"Archivos corruptos o no legibles: {df_a['is_corrupt'].sum()}")
print(f"Resoluciones presentes: {df_a[['width', 'height']].drop_duplicates().to_dict('records')}")
print(f"Tipos de dato de matriz: {df_a['dtype'].unique()}")
print(f"Canales de color: {df_a['channels'].unique()}")

# Resumen estadístico de calidad (Brillo, Contraste RMS, Nitidez Laplaciana)
df_a.groupby("label")[["mean_brightness", "std_brightness", "laplacian_var"]].describe().T

## 2. Análisis Univariado (Comportamiento Individual de Variables)

Se examinan las distribuciones individuales de intensidad de píxeles, los mapas espaciales promedio y las firmas de textura LBP.

### 2.1 Distribución de Densidad de Píxeles (Grayscale / RGB)

In [ ]:
plt.figure(figsize=(12, 5))

# Histograma de intensidad en escala de grises por clase
plt.subplot(1, 2, 1)
for label, color in [("Positive", "red"), ("Negative", "green")]:
    sample_ids = df_a[df_a["label"] == label]["image_id"].sample(30, random_state=42)
    pixels = [cv2.cvtColor(images_dict[img_id], cv2.COLOR_RGB2GRAY).ravel() for img_id in sample_ids]
    sns.kdeplot(np.concatenate(pixels), label=f"Clase {label}", color=color, fill=True, alpha=0.3)

plt.title("Densidad de Intensidad de Píxeles (Escala de Grises)")
plt.xlabel("Valor de Píxel (0-255)")
plt.ylabel("Densidad")
plt.legend()

# Histograma de varianza de brillo
plt.subplot(1, 2, 2)
sns.histplot(data=df_a, x="mean_brightness", hue="label", kde=True, palette=["#e74c3c", "#2ecc71"])
plt.title("Distribución de Brillo Medio por Imagen")
plt.xlabel("Brillo Medio")
plt.ylabel("Frecuencia")

plt.tight_layout()
plt.show()

### 2.2 Comportamiento Espacial Promedio

In [ ]:
# Calcular imagen promedio para Positive y Negative
pos_imgs = [images_dict[img_id] for img_id in df_a[df_a["label"] == "Positive"]["image_id"]]
neg_imgs = [images_dict[img_id] for img_id in df_a[df_a["label"] == "Negative"]["image_id"]]

mean_pos = np.mean(pos_imgs, axis=0).astype(np.uint8)
mean_neg = np.mean(neg_imgs, axis=0).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(mean_pos)
axes[0].set_title("Mapa Espacial Promedio: Positive (Con Grieta)")
axes[0].axis("off")

axes[1].imshow(mean_neg)
axes[1].set_title("Mapa Espacial Promedio: Negative (Sin Grieta)")
axes[1].axis("off")

diff_map = np.abs(mean_pos.astype(int) - mean_neg.astype(int)).astype(np.uint8)
axes[2].imshow(cv2.cvtColor(diff_map, cv2.COLOR_RGB2GRAY), cmap="hot")
axes[2].set_title("Diferencia Absoluta Espacial")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### 2.3 Diversidad de Textura (Local Binary Patterns - LBP)

In [ ]:
radius = 1
n_points = 8 * radius

def compute_lbp_hist(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lbp = local_binary_pattern(gray, n_points, radius, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
    return hist / hist.sum()

pos_lbp = np.mean([compute_lbp_hist(img) for img in pos_imgs[:50]], axis=0)
neg_lbp = np.mean([compute_lbp_hist(img) for img in neg_imgs[:50]], axis=0)

plt.figure(figsize=(8, 4))
x = np.arange(len(pos_lbp))
width = 0.35

plt.bar(x - width/2, pos_lbp, width, label="Positive (Con Grieta)", color="#e74c3c")
plt.bar(x + width/2, neg_lbp, width, label="Negative (Sin Grieta)", color="#2ecc71")
plt.title("Firma de Textura LBP (Local Binary Pattern) Promedio")
plt.xlabel("Patrones LBP Uniformes")
plt.ylabel("Frecuencia Normalizada")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Formulación de Hipótesis y Análisis Bivariado / Multivariado

### Hipótesis (1)
**Enunciado**: *Las imágenes con grietas presentan mayor varianza en los bordes y gradientes de color que las imágenes sin grieta.*

**Metodología de validación**: Extracción de respuesta de filtro Sobel y Canny por imagen. Comparación de distribuciones entre clases mediante prueba t de Student / Mann-Whitney U.

In [ ]:
sobel_vars = []
canny_means = []

for img_id in df_a["image_id"]:
    gray = cv2.cvtColor(images_dict[img_id], cv2.COLOR_RGB2GRAY)
    
    # Sobel Gradient Variance
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobelx**2 + sobely**2)
    sobel_vars.append(np.var(sobel_mag))
    
    # Canny Edge Mean
    edges = cv2.Canny(gray, 50, 150)
    canny_means.append(np.mean(edges))

df_a["sobel_var"] = sobel_vars
df_a["canny_mean"] = canny_means

# Gráficos bivariados
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=df_a, x="label", y="sobel_var", ax=axes[0], palette=["#e74c3c", "#2ecc71"])
axes[0].set_title("Varianza del Magnitud de Sobel por Clase")
axes[0].set_ylabel("Varianza Gradiente Sobel")

sns.boxplot(data=df_a, x="label", y="canny_mean", ax=axes[1], palette=["#e74c3c", "#2ecc71"])
axes[1].set_title("Respuesta Media de Bordes Canny por Clase")
axes[1].set_ylabel("Densidad Media de Bordes Canny")

plt.tight_layout()
plt.show()

# Prueba de Hipótesis Estadística (Mann-Whitney U)
pos_sobel = df_a[df_a["label"] == "Positive"]["sobel_var"]
neg_sobel = df_a[df_a["label"] == "Negative"]["sobel_var"]
u_stat, p_val = stats.mannwhitneyu(pos_sobel, neg_sobel)

print("--- RESULTADOS PRUEBA ESTADÍSTICA HIPÓTESIS 1 ---")
print(f"Estadístico Mann-Whitney U: {u_stat:.4f}")
print(f"p-value: {p_val:.4e}")
if p_val < 0.05:
    print("Conclusión: Se RECHAZA la hipótesis nula. Las imágenes con grietas presentan una varianza de bordes significativamente MAYOR (p < 0.05).")
else:
    print("Conclusión: No hay evidencia estadística suficiente para rechazar la hipótesis nula.")

### Hipótesis (2)
**Enunciado**: *El brillo general de la foto no afecta la diferencia cromática entre las clases.*

**Metodología de validación**: Cálculo del espacio cromático CIELAB (canales a* y b*) y matriz de correlación entre brillo medio y diferencias de color por etiqueta.

In [ ]:
chroma_stds = []

for img_id in df_a["image_id"]:
    lab = cv2.cvtColor(images_dict[img_id], cv2.COLOR_RGB2LAB)
    a_chan = lab[:, :, 1]
    b_chan = lab[:, :, 2]
    chroma_std = np.std(a_chan) + np.std(b_chan)
    chroma_stds.append(chroma_std)

df_a["chroma_std"] = chroma_stds

# Matriz de correlación
corr_vars = ["class_id", "mean_brightness", "std_brightness", "chroma_std", "sobel_var"]
corr_matrix = df_a[corr_vars].corr(method="spearman")

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Matriz de Correlación de Spearman (Brillo vs Cromatismo vs Clase)")
plt.tight_layout()
plt.show()

# Evaluación de correlación entre Brillo y Cromatismo
corr_val, corr_p = stats.spearmanr(df_a["mean_brightness"], df_a["chroma_std"])
print("--- RESULTADOS EVALUACIÓN HIPÓTESIS 2 ---")
print(f"Correlación Spearman (Brillo vs Desviación Cromática): {corr_val:.4f} (p-val: {corr_p:.4e})")
if abs(corr_val) < 0.2:
    print("Conclusión: Hipótesis VALIDADA. El brillo general de la foto NO presenta una correlación sustancial con la variación cromática.")
else:
    print("Conclusión: El brillo sí impacta moderadamente las mediciones cromáticas.")

## 4. Conclusiones y Recomendaciones de Ingeniería

### 4.1 Preprocesamiento Requerido
1. **Normalización y Estandarización**: Escalar píxeles al rango $[0, 1]$ y estandarizar con media y desviación estándar de ImageNet ($\mu=[0.485, 0.456, 0.406]$, $\sigma=[0.229, 0.224, 0.225]$) para modelos convolucionales.
2. **Ecualización Adaptativa de Histograma (CLAHE)**: Aplicar CLAHE en el canal de luminancia $L$ del espacio LAB para homogenizar sombras e iluminación sin alterar las discontinuidades estructurales de la grieta.
3. **Filtro Bilateral o Mapeo de Ruido**: Aplicar filtrado bilateral suave para reducir el grano propio del concreto rugoso sin borrar los bordes agudos de las grietas.

### 4.2 Estrategia de Aumentación de Datos
- **Transformaciones Geométricas**: Rotaciones aleatorias ($90^\circ, 180^\circ, 270^\circ$) y volcados (*flips*) horizontales/verticales (preservan la invariancia rotacional de las grietas).
- **Variaciones de Iluminación**: `ColorJitter` con variaciones moderadas de brillo ($\pm 15\%$) y contraste ($\pm 15\%$).
- **Recortes Aleatorios (Random Crop)**: Recortes de tamaño $200 	imes 200$ re-escalados a $227 	imes 227$ para asegurar la invarianza a escala.